## Settings

In [1]:
## auto reload modules
%reload_ext autoreload
%autoreload 2

## Dependencies

In [2]:
## libraries
import sys
import pickle
from pathlib import Path

## path
root = Path.cwd().resolve().parent
sys.path.insert(0, str(root))

## modules
from src.data.builders import load_processed_data
from src.estimators.factories import load_estimators
from src.evaluators.caching import load_notebook_cache
from src.evaluators.resampling import (
    logo_cross_valid,
    kfold_cross_valid
)
from src.evaluators.transfering import (
    compile_domain_transfer,
    results_domain_transfer
)
from src.evaluators.predicting import (
    compile_prediction_consensus,
    results_prediction_consensus
)

## constants
from src.evaluators.config import (
    FEAT_X,
    FEAT_Z, 
    TARGET
)

## Initialization

In [3]:
## reproducibility
N_DECIMALS = 2
N_REPEATS = 30
RANDOM_STATE = 42
FORCE_RECOMPUTE = False

## load data and models
data = load_processed_data()
models = load_estimators(random_state = RANDOM_STATE)

## view data shape
n_obs, n_feat = data.shape
print(f"Original Data: {n_feat} features, {n_obs} observations")

## view model surface
n_mods = len(models)
print(f"Learned Models: {n_mods} estimators")

## cache setup
cache_path = root / "notebooks" / "cache" / "transfer_results.pkl"
cache_metadata = {
    "n_obs": len(data),
    "n_repeats": N_REPEATS,
    "random_state": RANDOM_STATE,
    "model_names": sorted(models.keys()),
    "target": TARGET,
    "feat_x": list(FEAT_X),
    "feat_z": list(FEAT_Z),
}
cache_keys = (
    "predicts_dict_domain",
    "predicts_dict_disc",
    "predicts_dict_5fold",
    "predicts_dict_10fold",
    "results_data_domain",
    "results_data_disc",
    "results_data_5fold",
    "results_data_10fold",
    "results_data_consensus",
)
cache_payload = load_notebook_cache(
    cache_path = cache_path,
    metadata = cache_metadata,
    required_keys = cache_keys,
    force_recompute = FORCE_RECOMPUTE,
)
if cache_payload is not None:
    globals().update({key: cache_payload[key] for key in cache_keys})

Original Data: 32 features, 25 observations
Learned Models: 9 estimators
Loaded results from /Users/nickkunz/Documents/Nick’s MacBook Pro/Python/graph-capacity/notebooks/cache/transfer_results.pkl


## Training

In [4]:
## leave-one-group-out cross validation (domain)
if cache_payload is None:
    results_dict_domain = dict()
    predicts_dict_domain = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, y_pred = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "domain",  ## fold on domain
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_domain[name] = frontier
        predicts_dict_domain[name] = y_pred

In [5]:
## leave-one-group-out cross validation (discipline)
if cache_payload is None:
    results_dict_disc = dict()
    predicts_dict_disc = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, y_pred = logo_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            group = "discipline",  ## fold on discipline
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_disc[name] = frontier
        predicts_dict_disc[name] = y_pred

In [6]:
## 5-fold cross validation
if cache_payload is None:
    results_dict_5fold = dict()
    predicts_dict_5fold = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, y_pred = kfold_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_splits = 5,  ## fold on random 80-20 split
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_5fold[name] = frontier
        predicts_dict_5fold[name] = y_pred

In [7]:
## 10-fold cross validation
if cache_payload is None:
    results_dict_10fold = dict()
    predicts_dict_10fold = dict()
    for name, model in models.items():
        print(f"Training {name}...")
        frontier, y_pred = kfold_cross_valid(
            data = data,
            feat_x = FEAT_X,
            feat_z = FEAT_Z,
            estimator_c = model.estimator_c,
            estimator_r = model.estimator_r,
            target = TARGET,
            n_splits = 10,  ## fold on random 90-10 split
            n_repeats = N_REPEATS,
            random_state = RANDOM_STATE
        )
        frontier["model"] = name
        results_dict_10fold[name] = frontier
        predicts_dict_10fold[name] = y_pred

## Post-Processing

In [8]:
## cache setup for transfer results
if cache_payload is None:

    ## compile domain-transfer results
    results_data_domain = compile_domain_transfer(results = results_dict_domain)
    results_data_disc = compile_domain_transfer(results = results_dict_disc)
    results_data_5fold = compile_domain_transfer(results = results_dict_5fold)
    results_data_10fold = compile_domain_transfer(results = results_dict_10fold)

    ## compile prediction consensus (domain logo)
    results_data_consensus = compile_prediction_consensus(
        predictions = predicts_dict_domain,
        data = data,
        target = TARGET,
        group = "domain"
    )

    ## persist canonical transfer results for visualization notebooks
    cache_path.parent.mkdir(parents = True, exist_ok = True)
    cache_payload = {
        "metadata": cache_metadata,
        "predicts_dict_domain": predicts_dict_domain,
        "predicts_dict_disc": predicts_dict_disc,
        "predicts_dict_5fold": predicts_dict_5fold,
        "predicts_dict_10fold": predicts_dict_10fold,
        "results_data_domain": results_data_domain,
        "results_data_disc": results_data_disc,
        "results_data_5fold": results_data_5fold,
        "results_data_10fold": results_data_10fold,
        "results_data_consensus": results_data_consensus,
    }
    cache_path.write_bytes(pickle.dumps(cache_payload))
    print(f"Cached transfer results -> {cache_path}")

## Domain Transfer Test
Four resampling regimes evaluate held-out-group transferability against random-split baselines. Leave-one-group-out (LOGO) cross-validation by domain and discipline assesses generalization to unseen groups; random 5-fold and 10-fold cross-validation establish reference baselines.

- **Domain (LOGO)**: Held-out across 5 domains; group splits fixed, learner initializations vary across repeats.
- **Discipline (LOGO)**: Held-out across 25 disciplines; group splits fixed, learner initializations vary across repeats.
- **Random (5-Fold)**: Repeated shuffled 80–20 splits.
- **Random (10-Fold)**: Repeated shuffled 90–10 splits.

### Reporting Convention
Each regime contributes one value per estimator, averaged across groups or folds and repeats. The summary table reports the median across estimators. `EI` is shown as median `[Q1–Q3]`; `VR`, `MV`, and `MS` report medians, with `MV` and `MS` on the log-target scale. Aggregations are equally weighted across groups, folds, repeats, and estimators.

In [9]:
## compile transfer bounds and baseline benchmarks
results = results_domain_transfer(
    {"Domain (LOGO)": results_data_domain, "Discipline (LOGO)": results_data_disc},
    {"Random (5-Fold)": results_data_5fold, "Random (10-Fold)": results_data_10fold},
    keys = ("Transfer", "Standard"),
    indicies = ("Evaluation", "Method"),
    n_repeats = N_REPEATS,
    random_state = RANDOM_STATE,
    decimals = N_DECIMALS
)

display(results)

Cross-Validation: 9 models, 30 repeats (seeds 42-71)
Across-model aggregation: median of model-level means
Resampling: LOGO splits are fixed across repeats; random k-fold splits are reshuffled across repeats
Weighting: groups, folds, repeats, and models are equally weighted; results are not observation-weighted


_Transfer `EI` is 0.77 (Domain LOGO) and 0.74 (Discipline LOGO), comparable to the random-split baselines (0.76–0.77). Violation rates differ across regimes (0.02–0.04) but remain low throughout, suggesting that estimator performance under strict held-out-group evaluation does not depart materially from random-partition baselines._

## Prediction Consensus Test

The same domain-level LOGO predictions evaluated for frontier efficiency above are scored here for prediction consensus: agreement between the predicted frontier values and the observed maxima of the withheld domain. Frontier estimation is scale-arbitrary across fitting methods, so consensus is measured by agreement in ordering and dependence rather than by residual norms.

- **Spearman (`ρ`)**: Global rank agreement between predicted and observed capacities.
- **Rank-Biased Overlap (`RBO`)**: Top-weighted agreement (persistence $p = 0.9$), emphasizing the high-capacity tail.
- **Distance Correlation (`dCor`)**: Nonlinear dependence between predictions and observations.
- **Consensus Index (`CI`)**: Geometric mean of agreement-scaled `ρ`, `RBO`, and `dCor`.

### Reporting Convention
Each held-out domain contributes one value per estimator. The summary table reports held-out domains row-wise. `CI` is shown as median `[Q1–Q3]` across estimators; component columns report medians. Prior to compounding, `ρ` is rescaled from `[-1, 1]` to `[0, 1]` as `(1 + ρ) / 2` so that negative rank disagreement lowers `CI` without collapsing to a single floor. Aggregations are equally weighted across estimators. Higher values indicate stronger consensus.

In [10]:
## prediction consensus by held-out domain
results_consensus = results_prediction_consensus(
    results = results_data_consensus,
    group_col = "group",
    index_name = "Domain",
    n_repeats = N_REPEATS,
    random_state = RANDOM_STATE,
    decimals = N_DECIMALS
)

display(results_consensus)

Cross-Validation: 9 models, 30 repeats (seeds 42-71)
Across-model aggregation: median across learners within held-out domains
Resampling: LOGO domain splits are fixed across repeats
Weighting: domains and models are equally weighted; results are not observation-weighted


,CI [IQR],ρ,RBO,dCor
Domain,,,,
Earth & Physical Sciences,"0.76 [0.66, 0.85]",0.60,0.87,0.61
Life Sciences & Medicine,"0.81 [0.76, 0.85]",0.50,0.85,0.78
Technology & Information,"0.88 [0.84, 0.95]",0.80,0.90,0.86
Trade & Institutions,"0.89 [0.86, 0.94]",0.80,0.94,0.78
Transportation & Infrastructure,"0.75 [0.69, 0.85]",0.30,0.86,0.69


_Prediction consensus is consistent across held-out domains. Median `CI` ranges from 0.75 to 0.89, with positive rank agreement (median `ρ` between 0.30 and 0.80) and elevated top-end agreement (`RBO` 0.85–0.94) across all domains. Consensus strength varies by domain but remains positive throughout, indicating that fitted frontiers preserve the observed capacity structure on domains withheld from training._